# Pharmacy Medicine Demand & Expiry-Risk Forecasting

Run this notebook in Google Colab to regenerate the included synthetic data, train the demand model, evaluate a 12-week holdout, forecast future demand, and estimate FEFO expiry exposure. All data is fictional; no patient or pharmacy records are used.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_DIR = Path('/content/Assignments')
PROJECT_DIR = REPO_DIR / 'pharmacy-medicine-demand-expiry-risk-forecasting'
if not PROJECT_DIR.exists():
    if not REPO_DIR.exists():
        subprocess.run([
            'git', 'clone', '--depth', '1',
            'https://github.com/nithinkumarreddy45/Assignments.git',
            str(REPO_DIR)
        ], check=True)
    if not PROJECT_DIR.exists():
        raise FileNotFoundError(f'Project folder not found: {PROJECT_DIR}')

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Project folder:', PROJECT_DIR)

## Generate data and run the forecasting pipeline

The pipeline creates 24 fictional SKUs with 104 weeks of generated demand, performs a chronological holdout evaluation, refits the model on all history, and writes forecast and expiry-risk evidence to the project folders.

In [ ]:
from src.generate_data import build_dataset
from src.pipeline import run_pipeline
import pandas as pd

build_dataset()
summary = run_pipeline()
pd.Series(summary, name='value').to_frame()

## Inspect evaluation and forecast output

In [ ]:
metrics = pd.read_csv(PROJECT_DIR / 'evidence/holdout_metrics_by_sku.csv')
forecasts = pd.read_csv(PROJECT_DIR / 'evidence/12_week_demand_forecast.csv')
expiry = pd.read_csv(PROJECT_DIR / 'evidence/expiry_risk_and_reorder.csv')

display(metrics.describe().round(3))
display(forecasts.head(12))
display(expiry.sort_values('risk_value_estimate', ascending=False).head(10))

## How to read the results

The backtest compares recursive 12-week forecasts against a baseline that repeats the recent four-week average. The included synthetic run showed only a modest WAPE improvement; this is not evidence of real-world performance. Expiry-risk estimates simulate first-expiry-first-out consumption and assume no replenishment during the forecast horizon. Treat the suggested reorder quantity as an educational estimate, not an automatic purchasing instruction.

AI assistance was used to draft parts of the project code and documentation. Review and understand the implementation before submitting it.